# randn-like-noise-source — worked example 1: randn_like preserves dtype, randn(*shape) does not

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `randn-like-noise-source`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`torch.randn_like(tensor)` produces standard-normal noise with the same shape, dtype, AND device as the input tensor. `torch.randn(*tensor.shape)` only matches the shape — it always returns `float32` on CPU regardless of the input's dtype. This distinction matters in mixed-precision training and on non-CPU devices, where a dtype mismatch causes errors or silent precision loss.

## Worked solution

**Step 1 — Create float64 input.** We create `sigma = torch.ones(4, 8, dtype=torch.float64)`. The dtype is explicitly float64.

**Step 2 — Sample noise with randn_like.** `eps_correct = torch.randn_like(sigma)` inherits shape `(4, 8)`, dtype `float64`, and device `cpu`. All three properties match `sigma`.

**Step 3 — Sample noise with randn(*shape) [the wrong way].** `eps_wrong = torch.randn(*sigma.shape)` produces shape `(4, 8)` but dtype `float32` — the dtype mismatch is silent at this line but will cause a type error when you try to compute `sigma * eps_wrong` in certain contexts.

**Step 4 — Compare.** We print both dtypes side by side to show the discrepancy. For VAE reparameterization, `mu + sigma * eps` must all be the same dtype, so `randn_like` is the only correct choice.

In [ ]:
import torch as t

t.manual_seed(50)

# Create sigma in float64 (common in double-precision research code)
sigma = t.ones(4, 8, dtype=t.float64)

# Correct: randn_like inherits dtype
eps_correct = t.randn_like(sigma)

# Wrong: randn(*shape) always float32
eps_wrong = t.randn(*sigma.shape)

print(f'sigma.dtype:       {sigma.dtype}')        # torch.float64
print(f'eps_correct.dtype: {eps_correct.dtype}')  # torch.float64  (matches)
print(f'eps_wrong.dtype:   {eps_wrong.dtype}')    # torch.float32  (mismatch!)
print(f'Shape matches: {eps_correct.shape == sigma.shape}')  # True

# Reparameterization with correct eps — no type error
mu = t.zeros(4, 8, dtype=t.float64)
z = mu + sigma * eps_correct
print(f'z.dtype: {z.dtype}')   # torch.float64 — dtype preserved end-to-end